<a href="https://colab.research.google.com/github/priyankabolem/PlantDiseasesDetection/blob/main/NewPlantDiseasesDataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Step 1: Download Dataset from Kaggle

In [2]:
from google.colab import files
files.upload()  # Upload kaggle.json when prompted

# Install and setup Kaggle CLI
!pip install -q kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download the dataset
!kaggle datasets download -d vipoooool/new-plant-diseases-dataset

# Extract dataset
import zipfile
with zipfile.ZipFile("new-plant-diseases-dataset.zip", 'r') as zip_ref:
    zip_ref.extractall("plant_disease_dataset")

print("✅ Dataset extracted successfully!")

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/vipoooool/new-plant-diseases-dataset
License(s): copyright-authors
100% 2.69G/2.70G [00:19<00:00, 204MB/s]
100% 2.70G/2.70G [00:19<00:00, 146MB/s]
✅ Dataset extracted successfully!


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Step 2: Load and Preprocess Dataset

In [6]:
!ls /content/plant_disease_dataset

'new plant diseases dataset(augmented)'  'New Plant Diseases Dataset(Augmented)'   test


In [4]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# ✅ Define Correct Paths
dataset_root = "/content/plant_disease_dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)"
train_dir = os.path.join(dataset_root, "train")
valid_dir = os.path.join(dataset_root, "valid")

# ✅ Check if paths exist
if not os.path.exists(train_dir) or not os.path.exists(valid_dir):
    raise FileNotFoundError(f"❌ Dataset directories not found. Check dataset extraction.")

# ✅ Image Data Preprocessing
img_size = (150, 150)
batch_size = 32

train_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical'
)

validation_generator = train_datagen.flow_from_directory(
    valid_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical'
)

# ✅ Fetch Class Indices
class_indices = train_generator.class_indices
index_to_label = {v: k for k, v in class_indices.items()}  # Reverse dictionary

print("✅ Dataset loaded successfully!")
print(f"✅ Class Indices: {class_indices}")

Found 70295 images belonging to 38 classes.
Found 17572 images belonging to 38 classes.
✅ Dataset loaded successfully!
✅ Class Indices: {'Apple___Apple_scab': 0, 'Apple___Black_rot': 1, 'Apple___Cedar_apple_rust': 2, 'Apple___healthy': 3, 'Blueberry___healthy': 4, 'Cherry_(including_sour)___Powdery_mildew': 5, 'Cherry_(including_sour)___healthy': 6, 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot': 7, 'Corn_(maize)___Common_rust_': 8, 'Corn_(maize)___Northern_Leaf_Blight': 9, 'Corn_(maize)___healthy': 10, 'Grape___Black_rot': 11, 'Grape___Esca_(Black_Measles)': 12, 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)': 13, 'Grape___healthy': 14, 'Orange___Haunglongbing_(Citrus_greening)': 15, 'Peach___Bacterial_spot': 16, 'Peach___healthy': 17, 'Pepper,_bell___Bacterial_spot': 18, 'Pepper,_bell___healthy': 19, 'Potato___Early_blight': 20, 'Potato___Late_blight': 21, 'Potato___healthy': 22, 'Raspberry___healthy': 23, 'Soybean___healthy': 24, 'Squash___Powdery_mildew': 25, 'Strawberry___Leaf_

Step 3: Build and Train CNN Model

In [5]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# ✅ Build CNN Model
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)),
    MaxPooling2D(2, 2),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(len(class_indices), activation='softmax')  # Auto adjust output neurons
])

# ✅ Compile Model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# ✅ Train Model
epochs = 10
model.fit(train_generator, validation_data=validation_generator, epochs=epochs)

# ✅ Save Model
model.save("/content/plant_disease_model.keras")
print("✅ Model training completed and saved successfully!")

/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 121s 53ms/step - accuracy: 0.3361 - loss: 2.3995 - val_accuracy: 0.8103 - val_loss: 0.6549
Epoch 2/10
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 113s 52ms/step - accuracy: 0.6954 - loss: 0.9719 - val_accuracy: 0.8421 - val_loss: 0.4838
Epoch 3/10
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 107s 49ms/step - accuracy: 0.7631 - loss: 0.7359 - val_accuracy: 0.8867 - val_loss: 0.3551
Epoch 4/10
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 103s 47ms/step - accuracy: 0.8076 - loss: 0.5874 - val_accuracy: 0.8760 - val_loss: 0.3902
Epoch 5/10
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 145s 49ms/step - accuracy: 0.8340 - loss: 0.4981 - val_accuracy: 0.9024 - val_loss: 0.3116
Epoch 6/10
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 104s 47ms/step - accuracy: 0.8588 - loss: 0.4279 - val_accuracy: 0.8990 - val_loss: 0.3230
Epoch 7/10
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 104s 47ms/step - accuracy: 0.8752 - loss: 0.3796 - val_accuracy: 0.8978 - val_loss: 0.3304
Epoch 8/10
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 106s 48ms/step - accuracy: 

In [7]:
# Save the trained model to Google Drive
model.save("/content/drive/MyDrive/plant_disease_model.keras")
print("✅ Model saved successfully in Google Drive!")

NameError: name 'model' is not defined

Step 4: Test the Model on a Sample Image

In [6]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import numpy as np
import os

# ✅ Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ✅ Step 2: Define Correct File Paths
model_path = "/content/drive/MyDrive/plant_disease_model.keras"  # Ensure correct path
sample_image_path = "/content/drive/MyDrive/Tomatodiseased.jpg"  # Ensure correct image name

# ✅ Step 3: Ensure the Model Exists
if not os.path.exists(model_path):
    raise FileNotFoundError(f"❌ Model file '{model_path}' not found. Check the path.")

# ✅ Step 4: Ensure the Image File Exists
if not os.path.exists(sample_image_path):
    raise FileNotFoundError(f"❌ Image file '{sample_image_path}' not found. Check the path.")

# ✅ Step 5: Load and Preprocess the Image
img = load_img(sample_image_path, target_size=(150, 150))  # Resize image to model input size
img_array = img_to_array(img) / 255.0  # Normalize image
img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension

# ✅ Step 6: Load Model and Make Predictions
model = tf.keras.models.load_model(model_path)
predictions = model.predict(img_array)

# ✅ Step 7: Decode Predictions
class_indices = {0: 'Healthy', 1: 'Diseased'}  # Replace with actual class labels
predicted_class = np.argmax(predictions, axis=1)[0]
predicted_label = class_indices.get(predicted_class, "Unknown")

# ✅ Step 8: Print the Prediction Result
print(f"🌿 Predicted class: {predicted_label}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


FileNotFoundError: ❌ Model file '/content/drive/MyDrive/plant_disease_model.keras' not found. Check the path.